# 01 — Synthetic Proof of Concept

**Goal**: Demonstrate that persistent homology on Takens-embedded light curves can
distinguish astrophysical signal from pure noise, even at sub-threshold SNR levels.

We generate synthetic light curves of four types:
- **Periodic** (sinusoidal variable star)
- **Transient** (supernova-like Bazin profile)
- **Stochastic** (AGN damped random walk)
- **Noise-only** (no astrophysical signal)

Each is generated at multiple SNR levels (1σ to 5σ) with LSST-like irregular cadence.
We embed each into phase space via Takens delay embedding, compute persistent homology,
and show that topological features (particularly H₁ loops for periodic sources) emerge
from the noise floor even at low SNR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from void.data.synthetic import (
    generate_periodic,
    generate_transient,
    generate_stochastic,
    generate_noise,
)
from void.embedding.takens import TakensEmbedder
from void.topology.persistence import compute_persistence
from void.viz.plots import (
    plot_light_curve,
    plot_embedding_2d,
    plot_persistence_diagram,
    plot_barcode,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
rng = np.random.default_rng(42)

## 1. Visual comparison: signal types at high SNR

First, let's see what each source type looks like with a clear signal (SNR=10).

In [ ]:
high_snr = 10.0
generators = {
    "periodic": lambda r: generate_periodic(snr=high_snr, period=30.0, n_epochs=300, rng=r),
    "transient": lambda r: generate_transient(snr=high_snr, n_epochs=300, rng=r),
    "stochastic": lambda r: generate_stochastic(snr=high_snr, tau=200, n_epochs=300, rng=r),
    "noise": lambda r: generate_noise(n_epochs=300, rng=r),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
high_snr_lcs = {}
for ax, (name, gen) in zip(axes, generators.items()):
    lc = gen(np.random.default_rng(rng.integers(0, 2**63)))
    high_snr_lcs[name] = lc
    plot_light_curve(lc, ax=ax, title=name.capitalize())
fig.suptitle("Source Types at SNR=10", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## 2. Takens embedding: phase space reconstruction

Embed each high-SNR light curve into 3D phase space. The periodic source should
produce a clear loop (torus cross-section); noise should be a featureless blob.

In [ ]:
embedder = TakensEmbedder(dimension=3, delay=2, interpolation="linear")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, lc) in zip(axes, high_snr_lcs.items()):
    cloud = embedder.embed(lc)
    plot_embedding_2d(cloud, ax=ax, title=f"{name.capitalize()} embedding")
fig.suptitle("Takens Embeddings (2D projection) at SNR=10", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## 3. Persistence diagrams at high SNR

Compute persistent homology (H₀ and H₁) on each embedded point cloud.
The periodic source should show a prominent H₁ feature (a long-lived loop).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, lc) in zip(axes, high_snr_lcs.items()):
    cloud = embedder.embed(lc)
    pd = compute_persistence(cloud, maxdim=1)
    plot_persistence_diagram(pd, ax=ax, title=f"{name.capitalize()}")
    total_h1 = pd.total_persistence(dim=1)
    ax.text(0.05, 0.95, f"H₁ total: {total_h1:.2f}",
            transform=ax.transAxes, fontsize=8, va="top")
fig.suptitle("Persistence Diagrams at SNR=10", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## 4. The key experiment: signal detection vs SNR

Now the critical test. Generate periodic sources at decreasing SNR (5σ down to 0.5σ)
and measure the total H₁ persistence. Compare to the distribution from pure noise.

**The question**: At what SNR does the topological signal become indistinguishable from noise?

In [ ]:
snr_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
n_trials = 30

# Collect H1 total persistence for periodic signals at each SNR
signal_persistence = {snr: [] for snr in snr_levels}
for snr in snr_levels:
    for trial in range(n_trials):
        trial_rng = np.random.default_rng(rng.integers(0, 2**63))
        lc = generate_periodic(snr=snr, period=30.0, n_epochs=300, rng=trial_rng)
        cloud = embedder.embed(lc)
        pd = compute_persistence(cloud, maxdim=1)
        signal_persistence[snr].append(pd.total_persistence(dim=1))

# Null distribution: pure noise
null_persistence = []
for trial in range(n_trials * 3):
    trial_rng = np.random.default_rng(rng.integers(0, 2**63))
    lc = generate_noise(n_epochs=300, rng=trial_rng)
    cloud = embedder.embed(lc)
    pd = compute_persistence(cloud, maxdim=1)
    null_persistence.append(pd.total_persistence(dim=1))

print(f"Null H₁ persistence: mean={np.mean(null_persistence):.3f}, "
      f"std={np.std(null_persistence):.3f}")
for snr in snr_levels:
    vals = signal_persistence[snr]
    print(f"SNR={snr:4.1f}: mean={np.mean(vals):.3f}, std={np.std(vals):.3f}")

In [ ]:
from void.viz.plots import plot_snr_comparison

fig = plot_snr_comparison(
    signal_persistence,
    null_values=null_persistence,
    title="Periodic Signal: H₁ Persistence vs Peak SNR",
)
plt.show()

## 5. Multi-source comparison across types

Compare all source types at the same sub-threshold SNR to see which
topological features distinguish them.

In [ ]:
sub_threshold_snr = 3.0
source_types = {
    "periodic": lambda r: generate_periodic(snr=sub_threshold_snr, period=30.0, n_epochs=300, rng=r),
    "transient": lambda r: generate_transient(snr=sub_threshold_snr, n_epochs=300, rng=r),
    "stochastic": lambda r: generate_stochastic(snr=sub_threshold_snr, tau=200, n_epochs=300, rng=r),
    "noise": lambda r: generate_noise(n_epochs=300, rng=r),
}

type_persistence = {name: {"total_h0": [], "total_h1": [], "entropy_h1": [], "n_feat_h1": []}
                    for name in source_types}

for name, gen in source_types.items():
    for trial in range(n_trials):
        trial_rng = np.random.default_rng(rng.integers(0, 2**63))
        lc = gen(trial_rng)
        cloud = embedder.embed(lc)
        pd = compute_persistence(cloud, maxdim=1)
        type_persistence[name]["total_h0"].append(pd.total_persistence(0))
        type_persistence[name]["total_h1"].append(pd.total_persistence(1))
        type_persistence[name]["entropy_h1"].append(pd.persistence_entropy(1))
        type_persistence[name]["n_feat_h1"].append(pd.n_features(1))

# Box plot comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ["total_h0", "total_h1", "entropy_h1", "n_feat_h1"]
metric_labels = ["Total H₀ Persistence", "Total H₁ Persistence",
                 "H₁ Entropy", "H₁ Feature Count"]

for ax, metric, label in zip(axes, metrics, metric_labels):
    data = [type_persistence[name][metric] for name in source_types]
    bp = ax.boxplot(data, labels=list(source_types.keys()), patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle(f"Topological Features by Source Type (SNR={sub_threshold_snr})", y=1.02)
fig.tight_layout()
plt.show()

## 6. Ensemble-level analysis

The core method: embed a *population* of sub-threshold sources into feature space
and compute persistent homology on the population point cloud.

Compare an ensemble with injected periodic signals to a pure-noise ensemble.

In [ ]:
from void.data.synthetic import generate_ensemble
from void.embedding.features import extract_features

# Ensemble with 50 periodic signals + 200 noise sources
signal_ensemble = generate_ensemble(
    n_signal=50, n_noise=200,
    signal_generator=generate_periodic,
    signal_kwargs={"snr": 3.0, "period": 30.0, "n_epochs": 200},
    noise_kwargs={"n_epochs": 200},
    rng=np.random.default_rng(123),
)

# Pure noise ensemble (null)
null_ensemble = generate_ensemble(
    n_signal=0, n_noise=250,
    noise_kwargs={"n_epochs": 200},
    rng=np.random.default_rng(456),
)

feat_embedder = TakensEmbedder(dimension=3, delay=2)

# Extract feature vectors for each source
signal_features = signal_ensemble.feature_matrix(
    lambda lc: extract_features(lc, embedder=feat_embedder, compute_tda=True)
)
null_features = null_ensemble.feature_matrix(
    lambda lc: extract_features(lc, embedder=feat_embedder, compute_tda=True)
)

print(f"Signal ensemble feature matrix: {signal_features.shape}")
print(f"Null ensemble feature matrix: {null_features.shape}")

In [ ]:
# Compute persistence on each ensemble's feature-space point cloud
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
signal_scaled = scaler.fit_transform(signal_features)
null_scaled = scaler.fit_transform(null_features)

pd_signal = compute_persistence(signal_scaled, maxdim=1)
pd_null = compute_persistence(null_scaled, maxdim=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_persistence_diagram(pd_signal, ax=ax1, title="Signal Ensemble (50 periodic + 200 noise)")
plot_persistence_diagram(pd_null, ax=ax2, title="Null Ensemble (250 noise)")

ax1.text(0.05, 0.85, f"H₁ total: {pd_signal.total_persistence(1):.2f}\n"
         f"H₁ features: {pd_signal.n_features(1)}",
         transform=ax1.transAxes, fontsize=9, va="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
ax2.text(0.05, 0.85, f"H₁ total: {pd_null.total_persistence(1):.2f}\n"
         f"H₁ features: {pd_null.n_features(1)}",
         transform=ax2.transAxes, fontsize=9, va="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
fig.tight_layout()
plt.show()

In [ ]:
# Feature space visualization: do signal sources cluster?
from void.viz.plots import plot_feature_space

# Labels: 1 for signal sources, 0 for noise
labels = np.array([1] * 50 + [0] * 200)

fig = plot_feature_space(
    signal_features,
    labels=labels,
    method="pca",
    title="Ensemble Feature Space (PCA) — Blue=noise, Yellow=periodic @ 3σ",
)
plt.show()

## Summary

Key findings from synthetic validation:

1. **Periodic signals produce distinctive H₁ features** (loops) in persistence diagrams
   that noise does not, even at SNR as low as ~2-3σ.

2. **Different source types have different topological signatures** — periodic sources
   are distinguished by H₁ persistence, transients by H₀ clustering, stochastic
   sources by intermediate entropy.

3. **Ensemble-level topology reveals population structure** — when we embed all sources
   in a region into feature space and compute persistence on that cloud, the presence
   of a sub-threshold population creates detectable topological features.

Next steps: Notebook 05 for rigorous null model calibration and power analysis.